# BQ10 — Revenue–Charge Gap có tạo được nhóm cần kiểm tra gói cước không?

## Câu hỏi kinh doanh
Với mỗi khách hàng:

$$RevenueChargeGap = MonthlyRevenue - TotalRecurringCharge$$

Khoảng gap nào tập trung churn, đủ nhiều khách hàng và ổn định để dùng cho bill/plan review? Gap có đủ bằng chứng để trở thành **reason code độc lập**, hay chỉ nên là **cờ phụ** cho khách đã có churn risk cao?

## Ba quyết định có thể có
- **REASON CODE CANDIDATE:** có risk, reach, stability và thông tin bổ sung rõ ràng.
- **AUXILIARY FLAG:** có ích để gợi ý kiểm tra hóa đơn/gói cước nhưng không đứng độc lập.
- **REMOVE:** không dùng gap cho chính sách giữ chân.

> Gap không phải margin, discount, overage hay bằng chứng khách đang trả sai gói. Muốn gọi là “plan mismatch” cần thêm dữ liệu cấu phần hóa đơn, gói cước và quyền lợi sử dụng.


In [ ]:
BQ10_VERSION = 'BQ10_ANALYSIS_FIRST_GAP_V2'
print(f'Đang chạy: {BQ10_VERSION}')

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, FuncFormatter
from IPython.display import display, Markdown
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import SplineTransformer

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42
N_SPLITS = 5
N_KNOTS = 5
MIN_EFFECT = 0.02
MIN_REACH = 0.05
MAX_TAIL_CHANGE_PP = 1.00
MIN_LOGLOSS_GAIN = 0.001
MIN_AUC_GAIN = 0.005
MIN_CONSISTENT_FOLDS = 4


## 1. Khóa cohort và tạo gap

Giữ các giá trị bằng 0 để audit; chỉ loại churn không hợp lệ, dữ liệu thiếu và `MonthlyRevenue`/`TotalRecurringCharge` âm.


In [ ]:
DATA_CANDIDATES = [Path('../data/cell2cell_complete_cases.csv'), Path('data/cell2cell_complete_cases.csv')]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Không tìm thấy data/cell2cell_complete_cases.csv')

required_columns = ['Churn', 'MonthlyRevenue', 'TotalRecurringCharge']
raw = pd.read_csv(DATA_PATH, usecols=required_columns)

def normalize_binary(series):
    values = series.astype('string').str.strip().str.lower()
    mapping = {'yes': 1, 'y': 1, 'true': 1, '1': 1, '1.0': 1,
               'no': 0, 'n': 0, 'false': 0, '0': 0, '0.0': 0}
    return values.map(mapping)

work = raw.copy()
work['Churn_bin'] = normalize_binary(work['Churn'])
work['MonthlyRevenue'] = pd.to_numeric(work['MonthlyRevenue'], errors='coerce')
work['TotalRecurringCharge'] = pd.to_numeric(work['TotalRecurringCharge'], errors='coerce')
valid_mask = (
    work[['Churn_bin', 'MonthlyRevenue', 'TotalRecurringCharge']].notna().all(axis=1)
    & work['MonthlyRevenue'].ge(0)
    & work['TotalRecurringCharge'].ge(0)
)
analysis = work.loc[valid_mask].copy()
analysis['Churn_bin'] = analysis['Churn_bin'].astype(int)
analysis['RevenueChargeGap'] = (
    analysis['MonthlyRevenue'] - analysis['TotalRecurringCharge']
)

gap_p01, gap_p99 = analysis['RevenueChargeGap'].quantile([.01, .99])
qa = pd.DataFrame({
    'Chỉ tiêu': ['Tổng đầu vào', 'Dòng bị loại', 'Cohort BQ10',
                 'Gap âm', 'Gap bằng 0', 'Gap dương', 'Gap P01', 'Gap P99'],
    'Giá trị': [len(work), int((~valid_mask).sum()), len(analysis),
                int(analysis['RevenueChargeGap'].lt(0).sum()),
                int(analysis['RevenueChargeGap'].eq(0).sum()),
                int(analysis['RevenueChargeGap'].gt(0).sum()), gap_p01, gap_p99],
})
display(qa)
display(analysis[['MonthlyRevenue', 'TotalRecurringCharge', 'RevenueChargeGap']].describe(
    percentiles=[.01, .10, .50, .90, .99]).T)


## 2. Gap âm, bằng 0 và dương đang kể điều gì?

- **Gap âm:** `MonthlyRevenue` thấp hơn `TotalRecurringCharge` trong snapshot.
- **Gap bằng 0:** hai trường bằng nhau chính xác; phải audit vì có thể là cấu trúc ghi nhận dữ liệu.
- **Gap dương:** `MonthlyRevenue` cao hơn `TotalRecurringCharge`.

Chưa trạng thái nào tự động đồng nghĩa với “khách trả quá nhiều” hoặc “gói không phù hợp”.


In [ ]:
analysis['GapSign'] = pd.Categorical(
    np.select([analysis['RevenueChargeGap'].lt(0), analysis['RevenueChargeGap'].eq(0),
               analysis['RevenueChargeGap'].gt(0)],
              ['Âm', 'Bằng 0', 'Dương'], default='Không xác định'),
    categories=['Âm', 'Bằng 0', 'Dương'], ordered=True,
)
overall_churn = analysis['Churn_bin'].mean()
total_churners = analysis['Churn_bin'].sum()
sign_summary = (
    analysis.groupby('GapSign', observed=True)
        .agg(Customers=('Churn_bin', 'size'), Churners=('Churn_bin', 'sum'),
             ChurnRate=('Churn_bin', 'mean'), MedianGap=('RevenueChargeGap', 'median'))
        .reset_index()
)
sign_summary['Reach'] = sign_summary['Customers'] / len(analysis)
sign_summary['ChurnerCoverage'] = sign_summary['Churners'] / total_churners
sign_summary['Lift'] = sign_summary['ChurnRate'] / overall_churn
display(sign_summary)

central_gap = analysis.loc[analysis['RevenueChargeGap'].between(gap_p01, gap_p99), 'RevenueChargeGap']
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.hist(central_gap, bins=60, color='#277DA1', edgecolor='white', alpha=.9)
ax.axvline(0, color='#C1121F', linestyle='--', linewidth=2, label='Gap = 0')
ax.set(title='Phân phối Revenue–Charge Gap trong vùng P01–P99',
       xlabel='MonthlyRevenue − TotalRecurringCharge', ylabel='Số khách hàng')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
ax.legend()
plt.show()


## 3. Tìm candidate quanh mốc 0 bằng ngưỡng từ dữ liệu

Thay vì tự đặt các mốc −5, 5 hay 20, BQ10 tách riêng `Gap = 0`, sau đó chia phần âm và phần dương thành ba khoảng theo phân vị. Các biên thực tế được in ra để người đọc biết chính xác mỗi nhóm gồm ai. Đây vẫn là ngưỡng mô tả, không phải rule liên hệ khách hàng.

Các tỷ lệ được tính giống BQ09: **churn rate = churner / khách trong nhóm**, **reach = khách trong nhóm / toàn cohort**, **churner coverage = churner trong nhóm / toàn bộ churner**.


In [ ]:
negative = analysis.loc[analysis['RevenueChargeGap'].lt(0), 'RevenueChargeGap']
positive = analysis.loc[analysis['RevenueChargeGap'].gt(0), 'RevenueChargeGap']
if negative.empty or positive.empty:
    raise ValueError('Cần có cả gap âm và gap dương để tạo cửa sổ BQ10')
neg_q1, neg_q2 = negative.quantile([1/3, 2/3])
pos_q1, pos_q2 = positive.quantile([1/3, 2/3])

GAP_ORDER = ['N3 — Âm lớn', 'N2 — Âm vừa', 'N1 — Âm gần 0',
             'Z — Bằng 0', 'P1 — Dương gần 0', 'P2 — Dương vừa', 'P3 — Dương lớn']
def assign_gap_band(gap):
    labels = np.select(
        [gap.lt(neg_q1), gap.ge(neg_q1) & gap.lt(neg_q2),
         gap.ge(neg_q2) & gap.lt(0), gap.eq(0),
         gap.gt(0) & gap.le(pos_q1), gap.gt(pos_q1) & gap.le(pos_q2), gap.gt(pos_q2)],
        GAP_ORDER, default='Không xác định')
    if np.any(labels == 'Không xác định'):
        raise ValueError('Có giá trị gap chưa được gán nhóm')
    return pd.Categorical(labels, categories=GAP_ORDER, ordered=True)

analysis['GapBand'] = assign_gap_band(analysis['RevenueChargeGap'])
band_summary = (
    analysis.groupby('GapBand', observed=True)
        .agg(Customers=('Churn_bin', 'size'), Churners=('Churn_bin', 'sum'),
             ChurnRate=('Churn_bin', 'mean'), GapMin=('RevenueChargeGap', 'min'),
             GapMax=('RevenueChargeGap', 'max'), MedianGap=('RevenueChargeGap', 'median'),
             MedianRevenue=('MonthlyRevenue', 'median'),
             MedianCharge=('TotalRecurringCharge', 'median'))
        .reset_index()
)
band_summary['Reach'] = band_summary['Customers'] / len(analysis)
band_summary['ChurnerCoverage'] = band_summary['Churners'] / total_churners
band_summary['Lift'] = band_summary['ChurnRate'] / overall_churn
band_summary['Difference_pp'] = 100 * (band_summary['ChurnRate'] - overall_churn)
band_summary['ExcessChurners'] = (
    band_summary['Churners'] - band_summary['Customers'] * overall_churn
)
eligible_candidate = band_summary.loc[band_summary['Reach'].ge(MIN_REACH)]
candidate_index = eligible_candidate['ChurnRate'].idxmax()
candidate = band_summary.loc[candidate_index]
display(band_summary)
print(f'Candidate mô tả: {candidate["GapBand"]}')


## 4. Audit riêng `Gap = 0`

Nếu nhiều khách có hai trường bằng nhau chính xác và tập trung tại vài mức phí, đây có thể là cấu trúc dữ liệu/gói cước. Vì vậy `Gap = 0` không được tự động coi là hành vi khách hàng.


In [ ]:
exact_zero = analysis.loc[analysis['RevenueChargeGap'].eq(0)].copy()
if len(exact_zero):
    zero_audit = (
        exact_zero.groupby('TotalRecurringCharge')
            .agg(Customers=('Churn_bin', 'size'), Churners=('Churn_bin', 'sum'),
                 ChurnRate=('Churn_bin', 'mean'))
            .reset_index()
            .sort_values('Customers', ascending=False)
    )
    zero_audit['ShareWithinExactZero'] = zero_audit['Customers'] / len(exact_zero)
    zero_top5_concentration = zero_audit.head(5)['ShareWithinExactZero'].sum()
    display(zero_audit.head(15))
    print(f'Gap = 0: {len(exact_zero):,} khách; 5 mức phổ biến nhất chiếm '
          f'{zero_top5_concentration:.1%} nhóm exact-zero.')
else:
    zero_audit = pd.DataFrame()
    zero_top5_concentration = np.nan
    print('Không có khách hàng Gap = 0.')


## 5. Candidate có ổn định khi bỏ hai đuôi phân phối không?

Giữ nguyên các ngưỡng đã tạo, sau đó tính lại candidate khi chỉ dùng vùng P01–P99. Nếu churn gap thay đổi mạnh hoặc đảo chiều, candidate chưa đủ ổn định để hành động.


In [ ]:
candidate_band = candidate['GapBand']
candidate_mask = analysis['GapBand'].eq(candidate_band)
full_candidate_rate = analysis.loc[candidate_mask, 'Churn_bin'].mean()
full_other_rate = analysis.loc[~candidate_mask, 'Churn_bin'].mean()
full_difference_pp = 100 * (full_candidate_rate - full_other_rate)

central = analysis.loc[analysis['RevenueChargeGap'].between(gap_p01, gap_p99)].copy()
central_candidate_mask = central['GapBand'].eq(candidate_band)
central_candidate_rate = central.loc[central_candidate_mask, 'Churn_bin'].mean()
central_other_rate = central.loc[~central_candidate_mask, 'Churn_bin'].mean()
central_difference_pp = 100 * (central_candidate_rate - central_other_rate)
tail_change_pp = abs(central_difference_pp - full_difference_pp)
tail_is_stable = (np.sign(central_difference_pp) == np.sign(full_difference_pp)
                  and tail_change_pp <= MAX_TAIL_CHANGE_PP)

tail_sensitivity = pd.DataFrame({
    'Cohort': ['Toàn bộ dữ liệu', 'Chỉ P01–P99'],
    'CandidateCustomers': [int(candidate_mask.sum()), int(central_candidate_mask.sum())],
    'CandidateChurnRate': [full_candidate_rate, central_candidate_rate],
    'OtherChurnRate': [full_other_rate, central_other_rate],
    'Difference_pp': [full_difference_pp, central_difference_pp],
})
display(tail_sensitivity)
print(f'Thay đổi sau khi bỏ đuôi: {tail_change_pp:.2f} pp | Stable = {tail_is_stable}')


## 6. Guardrail kỹ thuật: gap có thêm thông tin ngoài hai biến gốc không?

Đây chỉ là kiểm tra phụ. Cross-validation so sánh mô hình spline dùng `MonthlyRevenue` + `TotalRecurringCharge` với mô hình thêm `RevenueChargeGap`. Nếu gap không cải thiện log loss và AUC nhất quán trên dữ liệu chưa thấy, gap không đủ tư cách làm reason code độc lập dù một nhóm mô tả có churn cao.


In [ ]:
def new_spline():
    return SplineTransformer(n_knots=N_KNOTS, degree=3, knots='quantile',
                             include_bias=False, extrapolation='linear')

y = analysis['Churn_bin'].to_numpy()
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
cv_rows = []
for fold, (train_index, valid_index) in enumerate(splitter.split(analysis, y), start=1):
    train = analysis.iloc[train_index]
    valid = analysis.iloc[valid_index]
    revenue_spline, charge_spline, gap_spline = new_spline(), new_spline(), new_spline()

    revenue_train = revenue_spline.fit_transform(train[['MonthlyRevenue']])
    charge_train = charge_spline.fit_transform(train[['TotalRecurringCharge']])
    gap_train = gap_spline.fit_transform(train[['RevenueChargeGap']])
    revenue_valid = revenue_spline.transform(valid[['MonthlyRevenue']])
    charge_valid = charge_spline.transform(valid[['TotalRecurringCharge']])
    gap_valid = gap_spline.transform(valid[['RevenueChargeGap']])

    base_train = np.column_stack([revenue_train, charge_train])
    base_valid = np.column_stack([revenue_valid, charge_valid])
    gap_model_train = np.column_stack([base_train, gap_train])
    gap_model_valid = np.column_stack([base_valid, gap_valid])

    base_model = LogisticRegression(C=100.0, solver='lbfgs', max_iter=2000, random_state=SEED)
    gap_model = LogisticRegression(C=100.0, solver='lbfgs', max_iter=2000, random_state=SEED)
    base_model.fit(base_train, y[train_index])
    gap_model.fit(gap_model_train, y[train_index])
    base_probability = base_model.predict_proba(base_valid)[:, 1]
    gap_probability = gap_model.predict_proba(gap_model_valid)[:, 1]

    base_loss = log_loss(y[valid_index], base_probability)
    gap_loss = log_loss(y[valid_index], gap_probability)
    base_auc = roc_auc_score(y[valid_index], base_probability)
    gap_auc = roc_auc_score(y[valid_index], gap_probability)
    cv_rows.append({'Fold': fold, 'BaseLogLoss': base_loss, 'GapLogLoss': gap_loss,
                    'LogLossGain': base_loss - gap_loss, 'BaseAUC': base_auc,
                    'GapAUC': gap_auc, 'AUCGain': gap_auc - base_auc})

cv_results = pd.DataFrame(cv_rows)
mean_logloss_gain = cv_results['LogLossGain'].mean()
mean_auc_gain = cv_results['AUCGain'].mean()
consistent_folds = int(((cv_results['LogLossGain'] > 0) & (cv_results['AUCGain'] > 0)).sum())
incremental_evidence = (mean_logloss_gain >= MIN_LOGLOSS_GAIN
                        and mean_auc_gain >= MIN_AUC_GAIN
                        and consistent_folds >= MIN_CONSISTENT_FOLDS)
display(cv_results)
display(pd.DataFrame([{'MeanLogLossGain': mean_logloss_gain, 'MeanAUCGain': mean_auc_gain,
                       'ConsistentFolds': consistent_folds,
                       'IncrementalEvidence': incremental_evidence}]))


## 7. Quyết định BQ10

Quyết định tách rõ bằng chứng mô tả và vai trò chính sách. Một candidate có risk/reach nhưng không có incremental evidence sẽ chỉ là **auxiliary flag**, không phải reason code độc lập.


In [ ]:
raw_excess = float(candidate['ChurnRate'] - overall_churn)
risk_is_meaningful = raw_excess >= MIN_EFFECT
reach_is_sufficient = float(candidate['Reach']) >= MIN_REACH
candidate_is_exact_zero = str(candidate_band).startswith('Z')

if (risk_is_meaningful and reach_is_sufficient and tail_is_stable
        and incremental_evidence and not candidate_is_exact_zero):
    decision = 'REASON CODE CANDIDATE'
    role = 'Có thể chuyển sang BQ19 để thiết kế thử nghiệm bill/plan review có treatment/control.'
elif risk_is_meaningful and reach_is_sufficient and tail_is_stable:
    decision = 'AUXILIARY FLAG'
    role = ('Chỉ gắn cờ kiểm tra hóa đơn/gói cước cho khách đã có churn risk cao; '
            'không liên hệ khách chỉ vì gap nằm trong candidate.')
else:
    decision = 'REMOVE'
    role = 'Không dùng RevenueChargeGap trong reason code hoặc rule giữ chân.'

evidence = pd.DataFrame({
    'Bằng chứng': ['Candidate', 'Khoảng gap quan sát', 'Số khách', 'Số churner',
                   'Churn rate', 'Chênh lệch với toàn cohort', 'Reach',
                   'Churner coverage', 'Lift', 'Độ nhạy khi bỏ đuôi',
                   'Mean log-loss gain', 'Mean AUC gain', 'Fold cùng cải thiện'],
    'Kết quả': [str(candidate_band),
                f'{candidate["GapMin"]:,.2f} đến {candidate["GapMax"]:,.2f}',
                f'{int(candidate["Customers"]):,}', f'{int(candidate["Churners"]):,}',
                f'{candidate["ChurnRate"]:.2%}', f'{100 * raw_excess:+.2f} pp',
                f'{candidate["Reach"]:.1%}', f'{candidate["ChurnerCoverage"]:.1%}',
                f'{candidate["Lift"]:.2f}x',
                f'{tail_change_pp:.2f} pp; stable={tail_is_stable}',
                f'{mean_logloss_gain:.5f}', f'{mean_auc_gain:+.4f}',
                f'{consistent_folds}/{N_SPLITS}'],
})
display(evidence)
print(f'QUYẾT ĐỊNH BQ10: {decision}')
print(role)


## 8. C10 — Biểu đồ kể chuyện chính

Panel trái cho thấy gap phân bố quanh 0. Panel phải đặt churn risk và quy mô cạnh nhau; màu vàng là candidate, không phải ngưỡng tự động can thiệp.


In [ ]:
fig, (ax_distribution, ax_risk) = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
ax_distribution.hist(central_gap, bins=60, color='#277DA1', edgecolor='white', alpha=.9)
ax_distribution.axvline(0, color='#C1121F', linestyle='--', linewidth=2, label='Gap = 0')
ax_distribution.axvspan(max(gap_p01, candidate['GapMin']),
                            min(gap_p99, candidate['GapMax']),
                            color='#F9C74F', alpha=.28, label='Candidate')
ax_distribution.set(title='Phân phối Revenue–Charge Gap (P01–P99)',
                    xlabel='MonthlyRevenue − TotalRecurringCharge', ylabel='Số khách hàng')
ax_distribution.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:,.0f}'))
ax_distribution.legend()

x = np.arange(len(band_summary))
bar_colors = ['#F9C74F' if i == candidate_index else '#90BE6D' for i in band_summary.index]
bars = ax_risk.bar(x, band_summary['Reach'], color=bar_colors, alpha=.85, label='Reach')
ax_risk.set_xticks(x, band_summary['GapBand'], rotation=25, ha='right')
ax_risk.set_ylabel('Reach')
ax_risk.yaxis.set_major_formatter(PercentFormatter(1))
ax_risk.bar_label(bars, labels=[f'{value:.1%}' for value in band_summary['Reach']], padding=3)
churn_axis = ax_risk.twinx()
churn_axis.plot(x, band_summary['ChurnRate'], color='#C1121F', marker='o',
                linewidth=2.5, label='Churn rate')
churn_axis.axhline(overall_churn, color='#555555', linestyle='--',
                   linewidth=1.3, label='Churn toàn cohort')
churn_axis.set_ylabel('Churn rate')
churn_axis.yaxis.set_major_formatter(PercentFormatter(1))
ax_risk.set_title('Risk và reach theo khoảng gap')
handles_1, labels_1 = ax_risk.get_legend_handles_labels()
handles_2, labels_2 = churn_axis.get_legend_handles_labels()
ax_risk.legend(handles_1 + handles_2, labels_1 + labels_2, loc='upper left')

fig.suptitle('BQ10 | Gap là reason code hay chỉ là cờ phụ?', fontsize=15, fontweight='bold')
decision_box = (f'{decision} | {candidate_band} | Risk {candidate["ChurnRate"]:.1%} | '
                f'Reach {candidate["Reach"]:.1%} | Incremental evidence={incremental_evidence}')
fig.text(0.5, -0.04, decision_box, ha='center', va='top', fontsize=11,
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#FFF4CC', edgecolor='#C9A227'))
plt.show()


## 9. Insight tự động để đưa vào báo cáo


In [ ]:
story = f'''
### Insight BQ10

Nhóm **{candidate_band}**, có gap từ **{candidate['GapMin']:,.2f} đến {candidate['GapMax']:,.2f}**, 
ghi nhận churn rate **{candidate['ChurnRate']:.1%}**, cao hơn toàn cohort 
**{100 * raw_excess:+.1f} điểm phần trăm**. Nhóm này bao phủ **{candidate['Reach']:.1%} khách hàng** 
({int(candidate['Customers']):,} người) và chứa **{candidate['ChurnerCoverage']:.1%} tổng churner**.

Khi bỏ hai đuôi P01–P99, chênh lệch candidate–các nhóm còn lại thay đổi 
**{tail_change_pp:.2f} điểm phần trăm**; kết quả stability là **{tail_is_stable}**. 
Kiểm tra ngoài mẫu cho thấy mean log-loss gain **{mean_logloss_gain:.5f}**, mean AUC gain 
**{mean_auc_gain:+.4f}**, với **{consistent_folds}/{N_SPLITS}** fold cùng cải thiện.

**Quyết định:** {decision}.  
**Vai trò giữ chân:** {role}

Gap chỉ là chênh lệch số học giữa hai feature đã có. Không gọi đây là margin, overage, discount 
hay plan mismatch đã được xác nhận nếu chưa có dữ liệu cấu phần hóa đơn và gói cước.
'''
display(Markdown(story))


## Nối BQ10 vào storytelling chung

1. **BQ08** xác định value proxy bằng `MonthlyRevenue`.
2. **BQ09** tìm vùng `TotalRecurringCharge` có risk và reach đáng chú ý.
3. **BQ10** kiểm tra chênh lệch revenue–charge có đủ tư cách làm reason code hay chỉ là cờ phụ.
4. Với **AUXILIARY FLAG**, chỉ khách đã có churn risk cao mới được kiểm tra bill/plan; không gửi ưu đãi đại trà.
5. Mọi can thiệp vẫn cần treatment/control để đo incremental churn reduction và revenue dilution.
